<a href="https://colab.research.google.com/github/ryouchinsa/sam3-cpp-macos/blob/master/notebooks/sam3_cpp_macos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segment Anything Model 3 CPP Wrapper for macOS and Ubuntu GPU

This code is to run a Segment Anything Model 3 ONNX model in c++ code and implemented on the macOS app RectLabel.

### Use GPU

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Runtime` -> `Change runtime type` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [1]:
!nvidia-smi

Thu Mar  5 15:24:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install packages

In [ ]:
!sudo apt-get update
!sudo apt-get install build-essential tar curl zip unzip autopoint libtool bison libx11-dev libxft-dev libxext-dev libxrandr-dev libxi-dev libxcursor-dev libxdamage-dev libxinerama-dev libxtst-dev cmake libgflags-dev libopencv-dev python3-dev

### Download ONNX Runtime

In [ ]:
!wget https://github.com/microsoft/onnxruntime/releases/download/v1.23.2/onnxruntime-linux-x64-gpu-1.23.2.tgz
!tar -xzvpf onnxruntime-linux-x64-gpu-1.23.2.tgz

### Install Segment Anything Model 3 CPP Wrapper

In [ ]:
!git clone https://github.com/ryouchinsa/sam3-cpp-macos.git

### Install SAM3

In [ ]:
!git clone https://github.com/facebookresearch/sam3.git

In [ ]:
!cp  sam3-cpp-macos/pyproject.toml sam3/

In [ ]:
%cd sam3

In [ ]:
!pip install -e .

In [ ]:
!pip install opencv-python onnx onnxruntime-gpu onnxsim matplotlib numba

### Download SAM 3 model from Hugging Face

In [ ]:
!hf auth login

In [ ]:
!hf download facebook/sam3 model.safetensors tokenizer.json

### Export ONNX models

This script is originated from [sam3-image](https://github.com/jamjamjon/usls/tree/main/scripts/sam3-image).

Edit --model-path according to your downloaded huggingface path.

If the exporting process is stopped, check being used RAM at the top-right corner.

In [ ]:
%cd /content/sam3-cpp-macos

In [ ]:
!python export.py --all --model-path /root/.cache/huggingface/hub/models--facebook--sam3/snapshots/3c879f39826c281e95690f02c7821c4de09afae7

In [ ]:
!cp /root/.cache/huggingface/hub/models--facebook--sam3/snapshots/3c879f39826c281e95690f02c7821c4de09afae7/tokenizer.json sam3

In [ ]:
!ls -l sam3

### Install tokenizers-cpp

In [ ]:
%cd /content/

In [ ]:
!git clone --recursive https://github.com/mlc-ai/tokenizers-cpp.git

In [ ]:
!cp /content/sam3-cpp-macos/tokenizers-cpp/CMakeLists.txt .
!cp /content/sam3-cpp-macos/tokenizers-cpp/msgpack/CMakeLists.txt msgpack
!cp /content/sam3-cpp-macos/tokenizers-cpp/sentencepiece/CMakeLists.txt sentencepiece

In [ ]:
!apt update
!apt install -y curl gcc make
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

In [ ]:
import os
os.environ['CARGO_HOME'] = '/root/.cargo'
os.environ['PATH'] = f"{os.environ['CARGO_HOME']}/bin:{os.environ['PATH']}"

In [ ]:
! rustc --version
! cargo --version

In [ ]:
%cd tokenizers-cpp/example

In [ ]:
!./build_and_run.sh

In [ ]:
%cd ..

In [ ]:
!mkdir lib
!cp ./example/build/tokenizers/sentencepiece/src/libsentencepiece.a lib/
!cp ./example/build/tokenizers/libtokenizers_c.a lib/
!cp ./example/build/tokenizers/libtokenizers_cpp.a lib/

### Build and run

In [ ]:
%cd /content/sam3-cpp-macos/

In [ ]:
!cmake -S . -B build -DONNXRUNTIME_ROOT_DIR=/content/onnxruntime-linux-x64-gpu-1.23.2 -DTOKENIZERS_ROOT_DIR=/content/tokenizers-cpp

In [ ]:
!cmake --build build

In [ ]:
!./build/sam3_cpp_test -vision_encoder="sam3/vision-encoder.onnx" -text_encoder="sam3/text-encoder.onnx" -geometry_encoder="sam3/geometry-encoder.onnx" -decoder="sam3/decoder.onnx" -tokenizer="sam3/tokenizer.json" -image="david-tomaseti-Vw2HZQ1FGjU-unsplash.jpg" -device="cuda:0" -text="zebra" -threshold=0.5

In [ ]:
!./build/sam3_cpp_test -vision_encoder="sam3/vision-encoder.onnx" -text_encoder="sam3/text-encoder.onnx" -geometry_encoder="sam3/geometry-encoder.onnx" -decoder="sam3/decoder.onnx" -tokenizer="sam3/tokenizer.json" -image="david-tomaseti-Vw2HZQ1FGjU-unsplash.jpg" -device="cuda:0" -boxes="pos:124,113,183,329" -threshold=0.5